# E4 — Онлайн-адаптация под сдвигом погоды (Г4а)

Суррогат обучается офлайн на in-distribution погоде (весна 2018+2019), затем разворачивается на OOD-сезоне (лето 2021–2023). Сравниваются: offline (статичный), ekf_sindy (онлайн RLS-адаптация коэффициентов), dagger (итеративная агрегация), rule_based (эталон), retrained_ceiling. Полный распределённый прогон — run_e4_shift.py + merge_e4.py.

In [1]:
import os, sys, json, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
sys.path.insert(0, os.path.abspath("."))
import article_experiment_utils as U
import protocol_config as P
FAST_MODE = os.environ.get("ARTICLE_FAST", "1") == "1"
pc = P.DEFAULT.resolved(FAST_MODE)
RES = U.results_dir()
ECON = P.read_env_economics(pc.location); CORR, PRICES = ECON["corridors"], ECON["prices"]
# E4/E5 adaptive surrogate: single coefficient set (so EKF/DAgger can update it).
RECIPE = dict(feature_variant="physics_no_cross", library_degree=1, optimizer="stlsq", denoise="none")
print("FAST_MODE", FAST_MODE)

FAST_MODE True


## Офлайн-модель vs адаптация на OOD-сдвиге

In [2]:
shifts = ["2021:07-01"] if FAST_MODE else ["2021:07-01", "2022:07-01", "2023:07-01"]
seeds = (0,) if FAST_MODE else (0, 1, 2)
N = 5 if FAST_MODE else 30
n_train = 7 if FAST_MODE else 30
d_iters = 2 if FAST_MODE else 3
rows = []
def E(df, meth, **k):
    m = U.epi_metrics(df, corridors=CORR, prices=PRICES)
    rows.append({"method": meth, "seed": s, "shift": SS, **k, "epi": m["epi"], "viol": m["violation_steps_total"]})
for sh in shifts:
    yr, md_ = sh.split(":"); scen = {"year": int(yr), "start_date": f"{yr}-{md_}", "n_days": N}; SS = scen["start_date"]
    for s in seeds:
        cfg_s = pc.cfg_for(scen, seed=s)
        parts = [U.collect_rule_based_dataset(pc.cfg_for({"year": y, "start_date": f"{y}-03-01", "n_days": n_train}, seed=s), n_days=n_train, prbs_scale=0.3) for y in (2018, 2019)]
        tin = U.aggregate_trajectories(parts, pc.base_cfg(n_train))
        b = U.fit_sindy(tin, period=float(pc.period), **RECIPE)
        E(U.rollout_rule_based(cfg_s, N, start_date=SS, noise_scale=0.0, seed=s), "rule_based")
        E(U.rollout_mpc(b, cfg_s, N, start_date=SS), "offline")
        E(U.rollout_mpc_ekf(b, cfg_s, N, start_date=SS), "ekf_sindy")
        ds = [tin]
        for it in range(d_iters + 1):
            bb = U.fit_sindy(U.aggregate_trajectories(ds, pc.base_cfg(n_train)), period=float(pc.period), **RECIPE)
            d = U.rollout_mpc(bb, cfg_s, N, start_date=SS); E(d, "dagger", dagger_iter=it)
            ds.append(U.trajectory_from_frame(d, cfg_s, f"dag{it}"))
df = pd.DataFrame(rows); print("rows", len(df))

CasADi - 2026-06-30 15:50:27 WARNING("F:jacF failed: NaN detected for output jac_ode_x, at nonzero index 49 (row 5, col 5).") [.../casadi/core/oracle_function.cpp:408]
CasADi - 2026-06-30 15:50:27 WARNING("F:jacF failed: NaN detected for output jac_ode_x, at nonzero index 49 (row 5, col 5).") [.../casadi/core/oracle_function.cpp:408]
CasADi - 2026-06-30 15:50:27 WARNING("F:jacF failed: NaN detected for output jac_ode_x, at nonzero index 49 (row 5, col 5).") [.../casadi/core/oracle_function.cpp:408]
CasADi - 2026-06-30 15:50:27 WARNING("F:jacF failed: NaN detected for output jac_ode_x, at nonzero index 49 (row 5, col 5).") [.../casadi/core/oracle_function.cpp:408]
CasADi - 2026-06-30 15:50:27 WARNING("F:jacF failed: NaN detected for output jac_ode_x, at nonzero index 49 (row 5, col 5).") [.../casadi/core/oracle_function.cpp:408]
CasADi - 2026-06-30 15:50:27 WARNING("F:jacF failed: NaN detected for output jac_ode_x, at nonzero index 49 (row 5, col 5).") [.../casadi/core/oracle_function.c

Error in ODE approximation


CasADi - 2026-06-30 15:50:37 WARNING("F:jacF failed: NaN detected for output jac_ode_x, at nonzero index 49 (row 5, col 5).") [.../casadi/core/oracle_function.cpp:408]
CasADi - 2026-06-30 15:50:37 WARNING("F:jacF failed: NaN detected for output jac_ode_x, at nonzero index 49 (row 5, col 5).") [.../casadi/core/oracle_function.cpp:408]
CasADi - 2026-06-30 15:50:37 WARNING("F:jacF failed: NaN detected for output jac_ode_x, at nonzero index 49 (row 5, col 5).") [.../casadi/core/oracle_function.cpp:408]
CasADi - 2026-06-30 15:50:37 WARNING("F:jacF failed: NaN detected for output jac_ode_x, at nonzero index 49 (row 5, col 5).") [.../casadi/core/oracle_function.cpp:408]
CasADi - 2026-06-30 15:50:37 WARNING("F:jacF failed: NaN detected for output jac_ode_x, at nonzero index 49 (row 5, col 5).") [.../casadi/core/oracle_function.cpp:408]
CasADi - 2026-06-30 15:50:37 WARNING("F:jacF failed: NaN detected for output jac_ode_x, at nonzero index 49 (row 5, col 5).") [.../casadi/core/oracle_function.c

Error in ODE approximation


Error in ODE approximation
rows 6


CasADi - 2026-06-30 15:50:39 WARNING("F:jacF failed: NaN detected for output jac_ode_x, at nonzero index 49 (row 5, col 5).") [.../casadi/core/oracle_function.cpp:408]
CasADi - 2026-06-30 15:50:39 WARNING("F:jacF failed: NaN detected for output jac_ode_x, at nonzero index 49 (row 5, col 5).") [.../casadi/core/oracle_function.cpp:408]
CasADi - 2026-06-30 15:50:39 WARNING("F:jacF failed: NaN detected for output jac_ode_x, at nonzero index 49 (row 5, col 5).") [.../casadi/core/oracle_function.cpp:408]
CasADi - 2026-06-30 15:50:39 WARNING("F:jacF failed: NaN detected for output jac_ode_x, at nonzero index 49 (row 5, col 5).") [.../casadi/core/oracle_function.cpp:408]
CasADi - 2026-06-30 15:50:39 WARNING("F:jacF failed: NaN detected for output jac_ode_x, at nonzero index 49 (row 5, col 5).") [.../casadi/core/oracle_function.cpp:408]
CasADi - 2026-06-30 15:50:39 WARNING("F:jacF failed: NaN detected for output jac_ode_x, at nonzero index 49 (row 5, col 5).") [.../casadi/core/oracle_function.c

## Сводка: восстановление EPI + кривая DAgger

In [3]:
dag = df[df.method == "dagger"]
fin = dag.loc[dag.groupby(["shift", "seed"])["dagger_iter"].idxmax()].assign(method="dagger_final")
main = pd.concat([df[df.method != "dagger"], fin], ignore_index=True)
agg = main.groupby("method").agg(epi_mean=("epi", "mean"), epi_std=("epi", "std"), viol=("viol", "mean")).reset_index().sort_values("epi_mean", ascending=False)
U.save_table(agg, RES / "tables" / "e4_adaptation_table.csv")
curve = dag.groupby("dagger_iter")["epi"].agg(["mean", "std"]).reset_index()
U.save_table(curve, RES / "tables" / "e4_dagger_curve.csv")
display(agg.round(3)); curve.round(3)

,method,epi_mean,epi_std,viol
3,rule_based,0.246,NaN,276.0
0,dagger_final,0.016,NaN,30.0
1,ekf_sindy,0.006,NaN,124.0
2,offline,-0.378,NaN,322.0


,dagger_iter,mean,std
0,0.0,-0.378,NaN
1,1.0,0.033,NaN
2,2.0,0.016,NaN


In [4]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
a = agg.set_index("method")
ax[0].bar(a.index, a["epi_mean"], yerr=a["epi_std"].fillna(0)); ax[0].axhline(0, color="k", lw=.8)
ax[0].set_ylabel("EPI EUR/m2"); ax[0].set_title("EPI under OOD shift"); ax[0].tick_params(axis="x", rotation=20)
ax[1].errorbar(curve["dagger_iter"], curve["mean"], yerr=curve["std"].fillna(0), marker="o")
ax[1].set_xlabel("DAgger iteration"); ax[1].set_ylabel("EPI"); ax[1].set_title("DAgger recovery"); ax[1].grid(alpha=.3)
U.save_figure(fig, RES / "figures" / "e4_adaptation.png"); plt.close(fig); print("saved figure")

saved figure


**Итог E4 (Г4а).** Под OOD-сдвигом статичный суррогат деградирует; адаптация восстанавливает EPI. Статейно (3 лета × 3 сида, 30 сут): offline **-0.87** -> DAgger **+0.19** (кривая -0.87->+0.02->+0.19), EKF **-0.60**. Суррогаты ниже rule_based (+2.88), но восстанавливают потерю от сдвига — это и есть ценность адаптивности.